# Spatial Panel Econometrics

This notebook covers:

1. Static spatial panels with fixed effects (Lee-Yu, Direct QML)
2. Dynamic spatial panels with bias correction
3. HAC-robust inference (Driscoll-Kraay, Conley)
4. Short-run and long-run spatial multipliers
5. Panel SEM
6. Formula interface for panels

> Panel data must be stacked as: units vary fastest within each time period: `[unit₁_t₁, unit₂_t₁, ..., unitN_t₁, unit₁_t₂, ...]`

In [ ]:
import Pkg
Pkg.activate(joinpath(@__DIR__, ".."))
Pkg.instantiate()

using SpatialEconometrics
using Random, LinearAlgebra, SparseArrays, Statistics

## 1. Generate Panel Data

We simulate a balanced spatial panel with:
- $N = 144$ regions (12×12 grid)
- $T = 8$ time periods
- Individual fixed effects $\mu_i$ and time effects $\xi_t$
- Spatial autoregressive parameter $\rho = 0.45$

In [ ]:
rng = MersenneTwister(123)
L = 12; N = L * L; T_per = 8; NT = N * T_per
W = make_rook_weights(L)

X = randn(rng, NT, 2)
β_true = [1.5, -0.8]
ρ_true = 0.45
μ_i = randn(rng, N)          # individual FE
ξ_t = randn(rng, T_per)      # time FE

S_lu = lu(sparse(1.0I, N, N) - ρ_true * W)
y = zeros(NT)
for t in 1:T_per
    idx = ((t - 1) * N + 1):(t * N)
    y[idx] = S_lu \ (μ_i .+ ξ_t[t] .+ X[idx, :] * β_true .+ 0.5 .* randn(rng, N))
end

println("Panel: N=$N regions, T=$T_per periods, NT=$NT observations")

## 2. Static Panel SAR — Lee-Yu Transformation

The Lee-Yu (2010) approach applies an orthogonal transformation to concentrate out
fixed effects from the likelihood, correcting the incidental parameters bias.

In [ ]:
const SP = SpatialEconometrics.StaticPanel

m_leeyu = fit_spatial_panel(SP.SAR(), SP.TwoWayFE(), W, X, y, N, T_per;
    approach = LeeYuTransformation(),
    varnames = ["investment", "trade"])
println(m_leeyu)
println("True ρ = $ρ_true → Estimated ρ̂ = $(round(m_leeyu.rho; digits=4))")

## 3. Direct QML with Bias Correction

Alternative: estimate directly and apply analytical bias correction (Hahn & Kuersteiner).

In [ ]:
m_dqml = fit_spatial_panel(SP.SAR(), SP.IndivFE(), W, X, y, N, T_per;
    approach = DirectQML(bias_correct=true),
    varnames = ["investment", "trade"])
println(m_dqml)
println("Lee-Yu ρ̂ = $(round(m_leeyu.rho; digits=4))")
println("DQML   ρ̂ = $(round(m_dqml.rho; digits=4))")

## 4. Spatial Effects from Panel Model

In [ ]:
effects = spatial_effects(m_leeyu)
println(effects)

## 5. HAC-Robust Inference

Standard errors may be invalid under heteroskedasticity or serial/spatial correlation.
**Driscoll-Kraay** corrects for cross-sectional dependence and serial correlation.
**Conley HAC** uses geographic distance for spatial correlation.

In [ ]:
# Driscoll-Kraay
dk = DriscollKraayVCOV(max_lag=3, kernel=BartlettKernel())
println(robust_coeftable(m_leeyu, dk))

In [ ]:
# Conley Spatial HAC
coords = hcat(repeat(1.0:L, L), repeat(1.0:L, inner=L))
conley = ConleySpatialHAC(coords, 3.0; max_time_lag=2)
println(robust_coeftable(m_leeyu, conley))

In [ ]:
# HAC-robust spatial effects
eff_robust = spatial_effects(m_leeyu, dk)
println("\nBaseline vs Driscoll-Kraay SEs:")
for i in 1:2
    println("  $(effects.varnames[i]) direct SE: baseline=$(round(effects.direct_se[i];digits=4)) → DK=$(round(eff_robust.direct_se[i];digits=4))")
end

## 6. Dynamic Spatial Panel

Adds a temporal lag $y_{t-1}$ and its spatial lag $Wy_{t-1}$:

$$y_t = \gamma y_{t-1} + \eta W y_{t-1} + \rho W y_t + X_t \beta + \mu_i + \varepsilon_{it}$$

Bias-corrected QML following Yu, de Jong & Lee (2008).

In [ ]:
const DP = SpatialEconometrics.DynamicPanel

# Generate dynamic panel data
T_dyn = 10; NT_dyn = N * T_dyn
X_dyn = randn(rng, NT_dyn, 1)
γ_true = 0.3; η_true = 0.15; ρ_dyn = 0.35

y_dyn = zeros(NT_dyn)
y_dyn[1:N] = S_lu \ (μ_i .+ X_dyn[1:N, :] * [0.8] .+ 0.4 .* randn(rng, N))
for t in 2:T_dyn
    idx = ((t - 1) * N + 1):(t * N)
    idx_prev = ((t - 2) * N + 1):((t - 1) * N)
    y_dyn[idx] = S_lu \ (μ_i .+ γ_true .* y_dyn[idx_prev] .+ η_true .* (W * y_dyn[idx_prev]) .+ X_dyn[idx, :] * [0.8] .+ 0.4 .* randn(rng, N))
end

m_dyn = fit_dynamic_spatial_panel(DP.SAR(), DP.IndivFE(), W, X_dyn, y_dyn, N, T_dyn;
    varnames=["productivity"])
println(m_dyn)

### Short-Run and Long-Run Multipliers

- **Short-run**: immediate spatial effect through $(I - \rho W)^{-1}$
- **Long-run**: equilibrium effect accounting for temporal persistence through $\gamma$ and $\eta$

In [ ]:
de = dynamic_spatial_effects(m_dyn)
println(de)

## 7. Panel SEM

In [ ]:
const SEM = SpatialEconometrics.SEMMod

# Generate SEM panel data
λ_true = 0.5
B_lu = lu(sparse(1.0I, N, N) - λ_true * W)
y_sem = zeros(NT)
for t in 1:T_per
    idx = ((t - 1) * N + 1):(t * N)
    u_t = B_lu \ (0.4 .* randn(rng, N))
    y_sem[idx] = μ_i .+ X[idx, :] * [1.5, -0.7] .+ u_t
end

m_sem = fit_spatial_error(SEM.TwoWayFE(), W, X, y_sem, N, T_per;
    varnames=["investment", "trade"])
println(m_sem)
println("True λ = $λ_true → Estimated λ̂ = $(round(m_sem.lambda; digits=4))")

## 8. Formula Interface for Panels

Use `@formula` with `slag()`, `fe()`, and `endo()` for panel estimation from tables.

In [ ]:
# Build table (data must have panel and time identifiers)
region_id = zeros(Int, NT); year_id = zeros(Int, NT)
for t in 1:T_per
    idx = ((t - 1) * N + 1):(t * N)
    region_id[idx] .= 1:N
    year_id[idx] .= 2020 + t
end
shuf = sortperm(1:NT; by = i -> (region_id[i], year_id[i]))
tbl = (region=region_id[shuf], year=year_id[shuf], y=y[shuf],
       investment=X[shuf, 1], trade=X[shuf, 2])

m_dsl = fit_spatial_panel(
    @formula(y ~ investment + trade + slag(investment + trade) + fe(region) + fe(year)),
    tbl, W; panel=:region, time=:year)
println(m_dsl)
println("Coefficient names: $(coefnames(m_dsl))")

## Summary

| Model | Function | Key Feature |
|-------|----------|-------------|
| Static SAR/SDM | `fit_spatial_panel` | Lee-Yu or Direct QML |
| Dynamic panel | `fit_dynamic_spatial_panel` | Bias correction, SR/LR effects |
| Panel SEM | `fit_spatial_error` | Spatial error structure |
| HAC-robust | `robust_vcov`, `robust_coeftable` | Driscoll-Kraay, Conley |
| Formula interface | `@formula` + `slag`, `fe` | Automatic FE detection |